In [11]:
import os
import shutil
from sklearn.model_selection import train_test_split
import numpy as np

def create_train_val_test_splits(main_data_dir, output_base_dir, train_size=0.7, val_size=0.15):
    """
    Create train, validation, and test splits by copying files to new directories.
    
    Parameters:
    main_data_dir: Original data directory containing class subdirectories
    output_base_dir: Where to create the split datasets
    train_size: Proportion for training (e.g., 0.7 for 70%)
    val_size: Proportion for validation (e.g., 0.15 for 15%)
    """
    # Validate inputs
    if train_size + val_size >= 1.0:
        raise ValueError("train_size + val_size must be less than 1.0")
    
    # Normalize paths and check for recursion
    main_data_dir = os.path.abspath(main_data_dir)
    output_base_dir = os.path.abspath(output_base_dir)
    
    if os.path.commonpath([main_data_dir]) == os.path.commonpath([main_data_dir, output_base_dir]):
        raise ValueError(f"Output directory '{output_base_dir}' cannot be inside or the same as input directory '{main_data_dir}'")
    
    # Create directories for splits
    splits = ['train', 'valid', 'test']
    split_dirs = {}
    
    # Start fresh - remove entire output directory if it exists
    if os.path.exists(output_base_dir):
        print(f"Removing existing output directory: {output_base_dir}")
        shutil.rmtree(output_base_dir)
    
    # Create base output directory
    os.makedirs(output_base_dir)
    
    # Create split directories
    for split in splits:
        split_dir = os.path.join(output_base_dir, split)
        split_dirs[split] = split_dir
        os.makedirs(split_dir)
    
    # Get all valid class directories - exclude any that match our split names
    classes = []
    for d in os.listdir(main_data_dir):
        dir_path = os.path.join(main_data_dir, d)
        if os.path.isdir(dir_path) and d not in splits:
            classes.append(d)
    
    if not classes:
        raise ValueError(f"No valid class directories found in {main_data_dir}")
    
    print(f"Found {len(classes)} classes: {classes}")
    
    # Create class subdirectories in each split
    for class_name in classes:
        for split in splits:
            os.makedirs(os.path.join(split_dirs[split], class_name))
    
    # Split and copy files for each class
    for class_name in classes:
        print(f"Processing class: {class_name}")
        
        # Get all files for this class
        class_dir = os.path.join(main_data_dir, class_name)
        files = [f for f in os.listdir(class_dir) 
                if os.path.isfile(os.path.join(class_dir, f)) and
                os.path.splitext(f)[1].lower() in ('.png', '.jpg', '.jpeg', '.tif', '.tiff')]
        
        # Skip empty classes
        if not files:
            print(f"  Warning: Class '{class_name}' has no valid image files. Skipping.")
            continue
        
        print(f"  Found {len(files)} image files")
        
        # First split: separate train+val from test
        train_val_files, test_files = train_test_split(
            files,
            train_size=train_size + val_size,
            random_state=42
        )
        
        # Second split: separate train from val
        train_files, val_files = train_test_split(
            train_val_files,
            train_size=train_size/(train_size + val_size),
            random_state=42
        )
        
        # Copy files to respective directories
        file_sets = [
            (train_files, 'train'),
            (val_files, 'valid'),
            (test_files, 'test')
        ]
        
        for file_list, target_split in file_sets:
            for fname in file_list:
                src = os.path.join(class_dir, fname)
                dst = os.path.join(split_dirs[target_split], class_name, fname)
                shutil.copy2(src, dst)
        
        # Print split sizes for this class
        print(f"  Train: {len(train_files)}")
        print(f"  Valid: {len(val_files)}")
        print(f"  Test: {len(test_files)}")
    
    return split_dirs

# Usage example:
if __name__ == "__main__":
    # Define your directories - MAKE SURE THESE ARE COMPLETELY SEPARATE
    main_data_dir = r"C:\Users\crumeike\Box\Midwest-Wave1-Data\Data"
    output_base_dir = r'C:\Users\crumeike\Documents\ROBINSON\damage-recognition\data'
    
    # Create splits
    try:
        split_dirs = create_train_val_test_splits(
            main_data_dir=main_data_dir,
            output_base_dir=output_base_dir,
            train_size=0.7,  # 70% for training
            val_size=0.15    # 15% for validation (remaining 15% for test)
        )
        print("Dataset splitting completed successfully!")
    except Exception as e:
        print(f"Error: {e}")

Found 6 classes: ['Debris', 'DS1-Slight', 'DS2-Moderate', 'DS3-Extensive', 'DS4-Complete', 'Undamaged']
Processing class: Debris
  Found 280 image files
  Train: 196
  Valid: 42
  Test: 42
Processing class: DS1-Slight
  Found 480 image files
  Train: 336
  Valid: 72
  Test: 72
Processing class: DS2-Moderate
  Found 512 image files
  Train: 358
  Valid: 77
  Test: 77
Processing class: DS3-Extensive
  Found 127 image files
  Train: 88
  Valid: 19
  Test: 20
Processing class: DS4-Complete
  Found 440 image files
  Train: 308
  Valid: 66
  Test: 66
Processing class: Undamaged
  Found 3744 image files
  Train: 2620
  Valid: 562
  Test: 562
Dataset splitting completed successfully!
